# POS rPPG — Step-by-Step Demonstration

This notebook walks through the **Plane-Orthogonal-to-Skin (POS)** algorithm of Wang et al. (2017) end-to-end, using the synthetic generator from `src/generator.py`. Each section pairs the math with executable code and a figure.

1. Build a synthetic RGB trace from the dichromatic reflection model where intensity drift and specular drift are ~10× the cardiac pulse.
2. Apply Algorithm 1 of the paper window-by-window.
3. Band-pass the recovered signal and read the BPM off the Welch PSD with parabolic peak interpolation.

**Expected outcome:** ~1% absolute BPM error vs. the 72 BPM ground truth despite the 10× distortion.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

# Make the project root importable from inside the notebooks/ folder.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.generator import SyntheticDataGenerator
from src.processor import POSProcessor, P_POS
from src.analyzer import SignalAnalyzer

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Generate the worst-case synthetic RGB trace

The dichromatic model (Eq. 6):

$$
\mathbf{C}(t) = I_0(1+i(t))\bigl(\mathbf{u}_c c_0 + \mathbf{u}_s s(t) + \mathbf{u}_p p(t)\bigr) + \mathbf{v}_n(t)
$$

with $\mathbf{u}_p = [0.33, 0.77, 0.53]$ (G > B > R) and $i(t), s(t)$ tuned so each contributes ~10× the per-channel pulse amplitude.

In [ ]:
FS = 30.0          # camera frame rate
DURATION = 30.0    # seconds
TRUE_BPM = 72.0

gen = SyntheticDataGenerator(
    fs=FS, duration_s=DURATION, pulse_bpm=TRUE_BPM,
    noise_to_pulse_ratio=10.0, rng=42,
)
data = gen.generate()
t, C = data["t"], data["C"]
print("RGB trace shape:", C.shape, "  fs =", FS, "Hz")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
for c, (col, lbl) in enumerate(zip(["#c0392b", "#27ae60", "#2980b9"], "RGB")):
    axes[0].plot(t, C[:, c], color=col, lw=0.8, label=lbl)
axes[0].set_title("Synthetic RGB trace — pulse is buried in intensity/specular drift")
axes[0].set_ylabel("Pixel intensity")
axes[0].legend(loc="upper right", ncol=3)

axes[1].plot(t, data["i"], color="#8e44ad", label="i(t)")
axes[1].plot(t, data["s"], color="#d35400", label="s(t)")
axes[1].plot(t, data["p"], color="#16a085", lw=1.4, label="p(t) (ground truth)")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Amplitude")
axes[1].legend(loc="upper right", ncol=3)
plt.tight_layout()

## 2. Inspect the POS projection matrix

$$P = \begin{pmatrix}0 & 1 & -1\\ -2 & 1 & 1\end{pmatrix}$$

Three checks before running the algorithm:

* $P\,\mathbf{1} = (0,0)^T$ — kills the common-mode intensity term.
* After temporal normalization, the pulse direction is $\mathbf{N}\mathbf{u}_p$ where $\mathbf{N} = \operatorname{diag}(1/(I_0\,\mathbf{u}_c c_0))$. $P\,(\mathbf{N}\mathbf{u}_p)$ should have **both entries positive** so $S_1, S_2$ are in-phase on the pulse and the `+` alpha-tuning is constructive.
* $P\,(\mathbf{N}\mathbf{u}_s)$ shows the residual specular direction that alpha-tuning has to remove.

In [ ]:
# Common-mode intensity is along 1 — POS kills it.
print("P @ [1,1,1]   =", P_POS @ np.array([1.0, 1.0, 1.0]))

# After temporal normalization, each component's direction is rescaled by
# N = diag(1 / (I0 * u_c * c0)). That's the vector POS actually sees.
I0, c0 = gen.I0, gen.c0
N = 1.0 / (I0 * data["u_c"] * c0)
u_p_n = N * data["u_p"]
u_s_n = N * data["u_s"]
print("P @ (N u_p)   =", P_POS @ u_p_n,
      "  -> both positive: pulse is in-phase on S_1 and S_2")
print("P @ (N u_s)   =", P_POS @ u_s_n,
      "  -> residual specular (alpha-tuning removes it)")

## 3. Run POS

Window length $l \approx 1.6\,f_s$ — about two cardiac cycles at rest.

In [ ]:
proc = POSProcessor(fs=FS)
H = proc.process(C)
print(f"window length l = {proc.window_samples} frames ({proc.window_samples/FS:.2f} s)")
print("H shape:", H.shape)

### 3.1 Peek inside one window

Pick window 100, walk through the four steps (normalize → project → tune → zero-mean), and plot $C_n$, $S_1$, $S_2$, $h$.

In [ ]:
k = 100
l = proc.window_samples
win = C[k:k+l]
Cn  = win / win.mean(axis=0, keepdims=True)
S   = Cn @ P_POS.T
S1, S2 = S[:, 0], S[:, 1]
alpha  = S1.std() / S2.std()
h      = S1 + alpha * S2
h      = h - h.mean()
print(f"alpha for window {k} = {alpha:.3f}")

tw = t[k:k+l]
fig, axes = plt.subplots(2, 2, figsize=(11, 5.5))
axes[0,0].plot(tw, Cn[:,0], "#c0392b"); axes[0,0].plot(tw, Cn[:,1], "#27ae60"); axes[0,0].plot(tw, Cn[:,2], "#2980b9")
axes[0,0].set_title("Step 1: temporally normalized RGB ($C_n$)")
axes[0,1].plot(tw, S1, "#c0392b", label="$S_1 = G_n - B_n$")
axes[0,1].plot(tw, S2, "#2980b9", label="$S_2 = G_n + B_n - 2R_n$")
axes[0,1].legend(); axes[0,1].set_title("Step 2: orthogonal projection")
axes[1,0].plot(tw, S1, "#c0392b", alpha=0.6, label="$S_1$")
axes[1,0].plot(tw, alpha*S2, "#2980b9", alpha=0.6, label=r"$\alpha\,S_2$")
axes[1,0].legend(); axes[1,0].set_title("Step 3: $\\alpha$-scaled $S_2$ vs $S_1$")
axes[1,1].plot(tw, h, "#16a085"); axes[1,1].set_title("Step 4: zero-meaned $h = S_1 + \\alpha S_2$")
for ax in axes.ravel(): ax.set_xlabel("Time [s]")
plt.tight_layout()

## 4. Filter, then read BPM off the spectrum

In [ ]:
an = SignalAnalyzer(fs=FS)
H_bp = an.bandpass(H)
res  = an.estimate_bpm(H_bp, method="welch")
cmp  = an.compare_to_ground_truth(res["bpm"], TRUE_BPM)
print(f"Detected: {cmp['estimated_bpm']:.2f} BPM   truth: {cmp['true_bpm']:.2f} BPM   |err|: {cmp['abs_error_bpm']:.2f} BPM ({cmp['rel_error_pct']:+.2f}%)")

In [ ]:
valid = l - 1
def z(x): x = x - x.mean(); s = x.std(); return x/s if s > 0 else x
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(t[valid:], z(data["p"][valid:]), color="#16a085", lw=1.6, label="ground truth p(t)")
ax.plot(t[valid:], z(H_bp[valid:]),     color="#c0392b", lw=1.2, label="recovered POS H(t)")
ax.set_xlabel("Time [s]"); ax.set_ylabel("z-score")
ax.set_title("Recovered POS pulse vs. ground truth (z-scored)")
ax.legend(); plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(res["freqs"]*60, res["spectrum"], color="#2c3e50")
ax.axvline(res["bpm"], color="#c0392b", ls="--", label=f"detected {res['bpm']:.2f} BPM")
ax.axvline(TRUE_BPM,   color="#16a085", ls=":",  label=f"truth {TRUE_BPM:.2f} BPM")
ax.set_xlim(0, 300); ax.set_xlabel("Frequency [BPM]"); ax.set_ylabel("Welch PSD")
ax.set_title("Spectrum of the recovered pulse")
ax.legend(); plt.tight_layout()

## 5. Robustness — seed sweep

Re-run the whole pipeline with 10 different RNG seeds to confirm the recovery is stable, not lucky.

In [ ]:
errors = []
for seed in [1, 7, 13, 42, 99, 123, 271, 314, 500, 777]:
    g = SyntheticDataGenerator(fs=FS, duration_s=DURATION, pulse_bpm=TRUE_BPM, noise_to_pulse_ratio=10.0, rng=seed)
    d = g.generate()
    H_k  = POSProcessor(fs=FS).process(d["C"])
    bpk  = SignalAnalyzer(fs=FS).estimate_bpm(SignalAnalyzer(fs=FS).bandpass(H_k))
    errors.append(abs(bpk["bpm"] - TRUE_BPM))
print(f"mean |err| = {np.mean(errors):.2f} BPM  |  max |err| = {np.max(errors):.2f} BPM")
print("per-seed:", [round(e,2) for e in errors])